# PrimeVul Score Conversion

Converts lm-eval generate_until output into router training data.
Same structure as convert_dataset_7_model.ipynb.

Paper Eq. 1: s_i^(t) = (1/M) * sum_m evaluate(y_hat, y)  — with repeats=5, score = fraction correct.

In [39]:
import json
import os
import glob
import random
import pandas as pd
from collections import defaultdict

random.seed(42)

output_file_path = "./datasets/split2_primevul"
os.makedirs(output_file_path, exist_ok=True)

# [org, model_name, output_dir_name]
# org/model_name  →  HuggingFace model ID used as key in scores dict
# output_dir_name →  directory name under output/primevul_gen_1to2/
model_list = [
    ["codellama",       "CodeLlama-7b-Instruct-hf",           "CodeLlama-7b-Instruct"],
    ["codellama",       "CodeLlama-13b-Instruct-hf",          "CodeLlama-13b-Instruct"],
    ["deepseek-ai",     "DeepSeek-Coder-V2-Lite-Instruct",    "DeepSeek-Coder-V2-Lite-Instruct"],
    ["Qwen",            "Qwen2.5-Coder-14B-Instruct",         "Qwen2.5-Coder-14B-Instruct"],
    ["bigcode",         "starcoder2-15b-instruct-v0.1",       "starcoder2-15b-instruct"],
    ["Virtue-AI-HUB",  "VulnLLM-R-7B",                       "VulnLLM-R-7B"],
]

# PrimeVul (generate_until)

In [40]:
import re

def extract_answer(raw: str) -> str:
    """Mirror lm-eval 'extract-answer': pull Yes/No from \\boxed{...}."""
    m = re.search(r"\\boxed\{(Yes|No)", raw, re.IGNORECASE)
    if m:
        return m.group(1).capitalize()
    m = re.search(r"\b(Yes|No)\b", raw, re.IGNORECASE)
    if m:
        return m.group(1).capitalize()
    return None   # malformed / unrecognised


output_data = []

for model_index, model_info in enumerate(model_list):
    model_pre  = model_info[0]
    model      = model_info[1]
    dir_name   = model_info[2]

    pattern = f"./output/primevul_gen_1to2/{dir_name}/*/samples_primevul_gen_1to2_*.jsonl"
    files = glob.glob(pattern)
    if not files:
        print(f"[skip] {model}: no samples file at {pattern}")
        continue

    doc_scores = {}   # doc_id -> float score
    doc_funcs  = {}   # doc_id -> func text

    with open(files[0]) as f:
        for line in f:
            if not line.strip():
                continue
            entry = json.loads(line)
            doc_id = entry["doc_id"]

            # All M=5 raw responses are in resps[0]
            resps = entry.get("resps", [[]])[0]
            target = entry["target"].strip()   # "Yes" or "No"

            # Paper Eq. 1:  s = (1/M) * sum  evaluate(y_hat_m, y)
            answers = [extract_answer(r) for r in resps]
            # print(f"doc_id={doc_id}  target={target}  answers={answers}")
            valid   = [a for a in answers if a is not None]
            if valid:
                correct = sum(1 for a in valid if a == target)
                acc = correct / len(resps)   # fraction of parseable runs correct
            else:
                acc = 0.0

            doc_scores[doc_id] = acc
            if doc_id not in doc_funcs:
                doc_funcs[doc_id] = entry["doc"]["func"]
            print(f"  -> {model} acc={acc:.3f} ({correct}/{len(valid)} valid)")

    sorted_doc_ids = sorted(doc_scores.keys())

    for i, doc_id in enumerate(sorted_doc_ids):
        acc = doc_scores[doc_id]
        if model_index == 0:
            output_data.append({
                "question": doc_funcs[doc_id],
                "scores":   {model_pre + "/" + model: acc}
            })
        else:
            output_data[i]["scores"][model_pre + "/" + model] = acc

    avg = sum(doc_scores.values()) / len(doc_scores)
    print(f"[ok] {model:<45} docs={len(sorted_doc_ids)}  avg_score={avg:.3f}")

print(f"\nTotal docs: {len(output_data)}")

  -> CodeLlama-7b-Instruct-hf acc=0.200 (1/5 valid)
  -> CodeLlama-7b-Instruct-hf acc=1.000 (5/5 valid)
  -> CodeLlama-7b-Instruct-hf acc=0.000 (0/5 valid)
  -> CodeLlama-7b-Instruct-hf acc=0.000 (0/5 valid)
  -> CodeLlama-7b-Instruct-hf acc=0.200 (1/5 valid)
  -> CodeLlama-7b-Instruct-hf acc=0.000 (0/5 valid)
  -> CodeLlama-7b-Instruct-hf acc=0.200 (1/5 valid)
  -> CodeLlama-7b-Instruct-hf acc=1.000 (5/5 valid)
  -> CodeLlama-7b-Instruct-hf acc=0.000 (0/5 valid)
  -> CodeLlama-7b-Instruct-hf acc=0.200 (1/5 valid)
  -> CodeLlama-7b-Instruct-hf acc=0.000 (0/5 valid)
  -> CodeLlama-7b-Instruct-hf acc=0.400 (2/5 valid)
  -> CodeLlama-7b-Instruct-hf acc=1.000 (5/5 valid)
  -> CodeLlama-7b-Instruct-hf acc=0.000 (0/5 valid)
  -> CodeLlama-7b-Instruct-hf acc=0.200 (1/5 valid)
  -> CodeLlama-7b-Instruct-hf acc=0.200 (1/5 valid)
  -> CodeLlama-7b-Instruct-hf acc=0.400 (2/5 valid)
  -> CodeLlama-7b-Instruct-hf acc=1.000 (5/5 valid)
  -> CodeLlama-7b-Instruct-hf acc=0.800 (4/5 valid)
  -> CodeLla

In [41]:
# Train / test split  70 / 30  (same as paper)
train_split_index = random.sample(range(len(output_data)), len(output_data))
output_data = [output_data[idx] for idx in train_split_index]

train_split = output_data[:int(0.7 * len(output_data))]
test_split  = output_data[int(0.7 * len(output_data)):]

with open(os.path.join(output_file_path, "primevul_train.json"), "w") as f:
    json.dump(train_split, f)

with open(os.path.join(output_file_path, "primevul_test.json"), "w") as f:
    json.dump(test_split, f)

print(f"train: {len(train_split)}  test: {len(test_split)}")
print(f"Saved to {output_file_path}/")

train: 12608  test: 5404
Saved to ./datasets/split2_primevul/


# Get ACC (per-model accuracy on full dataset)

In [42]:
with open(os.path.join(output_file_path, "primevul_train.json")) as f:
    output_data = json.load(f)

# Build score dict from model list
correct_dict = {m[0]+"/"+m[1]: 0 for m in model_list}

data_size = len(output_data)
for item in output_data:
    for key, score in item["scores"].items():
        if key in correct_dict:
            correct_dict[key] += score / data_size * 100

df = pd.DataFrame.from_dict(correct_dict, orient="index", columns=["accuracy (%)"])

# Also show how many queries have partial scores (0 < s < 1) per model
partial_dict = {m[0]+"/"+m[1]: 0 for m in model_list}
for item in output_data:
    for key, score in item["scores"].items():
        if key in partial_dict and 1e-9 < score < 1 - 1e-9:
            partial_dict[key] += 1

df["partial_scores"] = pd.Series(partial_dict)
df["partial_%"] = (df["partial_scores"] / data_size * 100).round(1)
df

,accuracy (%),partial_scores,partial_%
codellama/CodeLlama-7b-Instruct-hf,39.517766,5118,40.6
codellama/CodeLlama-13b-Instruct-hf,65.899429,2903,23.0
deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct,67.893401,1447,11.5
Qwen/Qwen2.5-Coder-14B-Instruct,70.923223,3490,27.7
bigcode/starcoder2-15b-instruct-v0.1,48.294734,5006,39.7
Virtue-AI-HUB/VulnLLM-R-7B,74.319480,1107,8.8


In [43]:
with open(os.path.join(output_file_path, "primevul_train.json")) as f:
    output_data = json.load(f)

M = 5   # number of runs per query

model_ids = list(output_data[0]["scores"].keys())
total_queries = len(output_data)

print(f"{'Model':<55} {'correct':>10} {'total':>10} {'accuracy':>10}")
print("─" * 90)

for model_id in model_ids:
    # score = correct / M  →  correct = score * M
    total_correct = sum(item["scores"][model_id] * M for item in output_data)
    total_possible = total_queries * M
    accuracy = total_correct / total_possible * 100
    print(f"{model_id:<55} {total_correct:>10.0f} {total_possible:>10,} {accuracy:>9.2f}%")

Model                                                      correct      total   accuracy
──────────────────────────────────────────────────────────────────────────────────────────
codellama/CodeLlama-7b-Instruct-hf                           24912     63,040     39.52%
codellama/CodeLlama-13b-Instruct-hf                          41543     63,040     65.90%
deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct                  42800     63,040     67.89%
Qwen/Qwen2.5-Coder-14B-Instruct                              44710     63,040     70.92%
bigcode/starcoder2-15b-instruct-v0.1                         30445     63,040     48.29%
Virtue-AI-HUB/VulnLLM-R-7B                                   46851     63,040     74.32%
